In [ ]:
# =============================================================================
# CELL 1 - Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.pca_plot.pca_plot_dataset as pca_plot_dataset_module
import src.pago_pipeline.pca_plot.pca_plot_render as pca_plot_render_module
import src.pago_pipeline.pca_plot.pca_plot_snapshot as pca_plot_snapshot_module
import src.pago_pipeline.pca_kmeans_snapshot as pca_kmeans_snapshot_module
from src.pago_pipeline.storage import sha256_of_file

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
pca_plot_dataset_module = importlib.reload(pca_plot_dataset_module)
pca_plot_render_module = importlib.reload(pca_plot_render_module)
pca_plot_snapshot_module = importlib.reload(pca_plot_snapshot_module)
pca_kmeans_snapshot_module = importlib.reload(pca_kmeans_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
load_latest_pca_kmeans_snapshot = (
    pca_kmeans_snapshot_module.load_latest_pca_kmeans_snapshot
)
resolve_pca_plot_snapshot = (
    pca_plot_snapshot_module.resolve_pca_plot_snapshot
)
latest_pca_plot_snapshot_is_available = (
    pca_plot_snapshot_module.latest_pca_plot_snapshot_is_available
)

In [ ]:
# =============================================================================
# CELL 2 - Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# =============================================================================
# CELL 3 - Define PCA plot snapshot configuration
# =============================================================================

PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "04-analysis" / "pca_kmeans"
)
PCA_PLOT_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "05-visualization" / "pca_plot"
)

PCA_KMEANS_SNAPSHOT_MODE = SnapshotMode.reuse_latest
PCA_PLOT_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
PLOT_DIMENSION_MODE = "auto"
PLOT_ROTATION_DEGREES = 225.0
PLOT_MIRROR_X_AXIS = True
OPEN_HTML_IN_BROWSER = False
UPDATE_LATEST_DIRECTORY = True

print(f"PCA/KMeans snapshot root directory: {PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA plot output root directory: {PCA_PLOT_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA plot snapshot mode: {PCA_PLOT_SNAPSHOT_MODE}")
print(f"Plot dimension mode: {PLOT_DIMENSION_MODE}")
print(f"Rotation degrees: {PLOT_ROTATION_DEGREES}")
print(f"Mirror X axis: {PLOT_MIRROR_X_AXIS}")
print(f"Open HTML in browser: {OPEN_HTML_IN_BROWSER}")

In [ ]:
# =============================================================================
# CELL 4 - Resolve active PCA/KMeans snapshot
# =============================================================================

pca_kmeans_snapshot_payload = load_latest_pca_kmeans_snapshot(
    snapshot_root_directory=PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY,
)

pca_kmeans_snapshot_directory = pca_kmeans_snapshot_payload["snapshot_directory"]
pca_kmeans_manifest_file_path = pca_kmeans_snapshot_payload["manifest_file_path"]
pca_kmeans_manifest_payload = pca_kmeans_snapshot_payload["manifest"]
cluster_assignments_file_path = pca_kmeans_snapshot_payload[
    "cluster_assignments_file_path"
]

print("Resolved PCA/KMeans snapshot successfully.")
print(f"Snapshot directory: {pca_kmeans_snapshot_directory}")
print(f"Cluster assignments path: {cluster_assignments_file_path}")

In [ ]:
# =============================================================================
# CELL 5 - Resolve active PCA plot snapshot
# =============================================================================

pca_plot_snapshot_payload = resolve_pca_plot_snapshot(
    snapshot_mode=PCA_PLOT_SNAPSHOT_MODE,
    snapshot_root_directory=PCA_PLOT_SNAPSHOT_ROOT_DIRECTORY,
    plot_dimension_mode=PLOT_DIMENSION_MODE,
    source_pca_kmeans_snapshot_root_directory=PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY,
    plot_rotation_degrees=PLOT_ROTATION_DEGREES,
    plot_mirror_x_axis=PLOT_MIRROR_X_AXIS,
    open_html_in_browser=OPEN_HTML_IN_BROWSER,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

pca_plot_snapshot_directory = pca_plot_snapshot_payload["snapshot_directory"]
pca_plot_manifest_file_path = pca_plot_snapshot_payload["manifest_file_path"]
pca_plot_manifest_payload = pca_plot_snapshot_payload["manifest"]
plot_data_file_path = pca_plot_snapshot_payload["plot_data_file_path"]
plot_html_file_path = pca_plot_snapshot_payload["plot_html_file_path"]
plot_profiling_log_file_path = pca_plot_snapshot_payload[
    "profiling_log_file_path"
]
plot_data_dataframe = pca_plot_snapshot_payload["plot_data"]
plot_profiling_log_dataframe = pca_plot_snapshot_payload["profiling_log"]

print("Resolved PCA plot snapshot successfully.")
print(f"Snapshot directory: {pca_plot_snapshot_directory}")
print(f"Plot data path: {plot_data_file_path}")
print(f"Interactive HTML path: {plot_html_file_path}")

In [ ]:
# =============================================================================
# CELL 6 - Print PCA plot snapshot summary
# =============================================================================

pca_plot_manifest_file_sha256 = sha256_of_file(
    input_file_path=pca_plot_manifest_file_path,
)
plot_data_file_sha256 = sha256_of_file(input_file_path=plot_data_file_path)
plot_html_file_sha256 = sha256_of_file(input_file_path=plot_html_file_path)

print("PCA plot snapshot is ready.")
print(
    f"Snapshot created at UTC: {pca_plot_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Plot data rows: {len(plot_data_dataframe)}")
print(f"Selected PCA component count: {pca_plot_manifest_payload['selected_pca_component_count']}")
print(f"Selected cluster count k: {pca_plot_manifest_payload['selected_cluster_count_k']}")
print(f"Selection reason: {pca_plot_manifest_payload['selection_reason']}")
print(f"Plot data SHA-256: {plot_data_file_sha256}")
print(f"HTML SHA-256: {plot_html_file_sha256}")
print(f"Manifest SHA-256: {pca_plot_manifest_file_sha256}")

In [ ]:
# =============================================================================
# CELL 7 - Inspect plot-ready dataframe and summaries
# =============================================================================

plot_dataframe_preview_row_limit = 10
plot_dataframe_preview = plot_data_dataframe.head(plot_dataframe_preview_row_limit).copy()
rendered_plot_dimension_count = int(
    pca_plot_manifest_payload.get("rendered_plot_dimension_count", 3)
)
plot_coordinate_column_names = ["plot_pc1", "plot_pc2"]
if "plot_pc3" in plot_dataframe_preview.columns:
    plot_coordinate_column_names.append("plot_pc3")

cluster_size_summary_dataframe = (
    plot_data_dataframe["cluster_label"]
    .value_counts()
    .rename_axis("cluster_label")
    .reset_index(name="row_count")
    .sort_values("cluster_label")
    .reset_index(drop=True)
)
pago_summary_dataframe = (
    plot_data_dataframe["pago_label"]
    .value_counts()
    .rename_axis("pago_label")
    .reset_index(name="row_count")
)

plot_preview_column_names = [
    "cluster_label",
    "description_label",
    "organism_label",
    "phylum_label",
    "class_label",
    "family_label",
    "genus_label",
    "pago_label",
] + plot_coordinate_column_names

print(f"Rendered plot dimension: {rendered_plot_dimension_count}D")
print("Plot-ready dataframe preview:")
display(plot_dataframe_preview[plot_preview_column_names])
print("Cluster size summary:")
display(cluster_size_summary_dataframe)
print("pAgo label summary:")
display(pago_summary_dataframe)


In [ ]:
# =============================================================================
# CELL 8 - Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- pca_kmeans_snapshot_directory")
print("- pca_kmeans_manifest_payload")
print("- pca_plot_snapshot_directory")
print("- pca_plot_manifest_payload")
print("- plot_data_file_path")
print("- plot_html_file_path")
print("- plot_profiling_log_file_path")
print("- plot_data_dataframe")
print("- plot_profiling_log_dataframe")
print("- cluster_size_summary_dataframe")
print("- pago_summary_dataframe")